# Driver drowsiness · MobileNetV3-Small

This notebook trains **two models**: UTA-RLDD alert/low-vigilance/drowsy and a
YawDD yawn/no-yawn video model. It contains the project code; no repository setup is needed.

**Before Run all:**
1. Select **Runtime → Change runtime type → T4 GPU** (or another CUDA GPU).
2. YawDD is read from your folder `1hRWkRD1xKQ0reWvM5o9OSGs1oPhCVSov`.
   Sign in with the Google account that has access; no public sharing is needed.
3. Run all cells and authorize Drive mounting in your own browser.

This smaller pipeline uses MRL Eyes (~342 MB) plus YawDD (~5.3 GB), avoiding
the 119.5 GB UTA-RLDD download and its public Google Drive quota.

Both datasets use subject-disjoint training, validation, and test splits.

YawDD labels supervise bags of mouth frames. They are not frame-level yawn
annotations. Scores on short live windows still need independent validation.
This notebook has not been GPU-executed by the coding assistant.


In [ ]:
"""Paste this entire file into the FIRST code cell of the existing Colab notebook.

Reads YawDD from your private folder using your own Colab login. No public sharing
or moving the original archive is necessary. Outputs still go to DriverDrowsiness.
"""
from pathlib import Path
import hashlib
import re
import shutil

YAWDD_FOLDER_ID = '1hRWkRD1xKQ0reWvM5o9OSGs1oPhCVSov'
EPOCHS = 40
BATCH_SIZE = 4


def find_yawdd(service, folder_id):
    if not re.fullmatch(r'[A-Za-z0-9_-]+', folder_id):
        raise ValueError('Set YAWDD_FOLDER_ID to the folder ID, not its full URL.')
    matches, page = [], None
    while True:
        result = service.files().list(
            q=f"'{folder_id}' in parents and name = 'YawDD.rar.gz' and trashed = false",
            fields='nextPageToken,files(id,name,size,md5Checksum,mimeType)',
            pageSize=100, pageToken=page, supportsAllDrives=True,
            includeItemsFromAllDrives=True).execute()
        matches.extend(result.get('files', []))
        page = result.get('nextPageToken')
        if not page:
            break
    if len(matches) != 1:
        raise RuntimeError(f'Found {len(matches)} files named YawDD.rar.gz in the specified folder. '
                           'Use the account that can access it and keep exactly one archive directly in that folder.')
    if not matches[0].get('size') or matches[0]['mimeType'] == 'application/vnd.google-apps.shortcut':
        raise RuntimeError('Expected the uploaded archive itself, not a Drive shortcut.')
    return matches[0]


def matches_archive(path, metadata):
    if not path.is_file() or path.stat().st_size != int(metadata['size']):
        return False
    with path.open('rb') as source:
        if source.read(2) != b'\x1f\x8b':
            return False
        if metadata.get('md5Checksum'):
            source.seek(0)
            digest = hashlib.md5(usedforsecurity=False)
            for block in iter(lambda: source.read(8 * 1024 * 1024), b''):
                digest.update(block)
            return digest.hexdigest() == metadata['md5Checksum']
    return True


def setup_colab():
    import torch
    from google.colab import drive, auth  # pyright: ignore[reportMissingImports]
    import google.auth  # pyright: ignore[reportMissingImports]
    from googleapiclient.discovery import build  # pyright: ignore[reportMissingImports]
    from googleapiclient.http import MediaIoBaseDownload  # pyright: ignore[reportMissingImports]

    if not torch.cuda.is_available():
        raise RuntimeError('Select Runtime > Change runtime type > T4 GPU before running this cell.')
    print('GPU:', torch.cuda.get_device_name(0))
    drive.mount('/content/drive')
    output_root = Path('/content/drive/MyDrive/DriverDrowsiness')
    output_root.mkdir(parents=True, exist_ok=True)
    target = Path('/content/driver_sources/YawDD.rar.gz')
    if (output_root / 'prepared_v1/yawn.zip').is_file():
        print('Completed YawDD crop cache found; preparation will restore it without downloading the archive.')
        return output_root, target

    # Authentication stays inside the user's Colab session. Never paste or share tokens.
    auth.authenticate_user()
    credentials, _ = google.auth.default()
    service = build('drive', 'v3', credentials=credentials, cache_discovery=False)
    metadata = find_yawdd(service, YAWDD_FOLDER_ID)
    print(f"Found {metadata['name']} ({int(metadata['size']) / 1e9:.2f} GB) in your selected folder.")
    if matches_archive(target, metadata):
        print('Reusing the verified local archive:', target)
        return output_root, target
    target.parent.mkdir(parents=True, exist_ok=True)
    if shutil.disk_usage(target.parent).free < int(metadata['size']) + 10_000_000_000:
        raise RuntimeError('Need archive size plus at least 10 GB free local disk space for preparation.')
    temporary = target.with_suffix('.gz.part')
    request = service.files().get_media(fileId=metadata['id'], supportsAllDrives=True)
    with temporary.open('wb') as destination:
        downloader = MediaIoBaseDownload(destination, request, chunksize=16 * 1024 * 1024)
        done, previous = False, -1
        while not done:
            status, done = downloader.next_chunk(num_retries=5)
            if status:
                percent = int(status.progress() * 100)
                if percent // 5 != previous or done:
                    print(f'YawDD download: {percent}%', flush=True)
                    previous = percent // 5
    if not matches_archive(temporary, metadata):
        raise RuntimeError('Downloaded archive failed size, gzip-header, or checksum validation. Rerun this cell.')
    temporary.replace(target)
    print('YawDD ready:', target)
    return output_root, target


if __name__ == '__main__':
    DRIVE_ROOT, YAWDD_PATH = setup_colab()


In [ ]:
import base64, io, os, zipfile
PROJECT = Path('/content/driver_project')
PROJECT.mkdir(exist_ok=True)
with zipfile.ZipFile(io.BytesIO(base64.b64decode('...'))) as archive:
    archive.extractall(PROJECT)
os.chdir(PROJECT)
print("Project restored:", PROJECT)


In [ ]:
import subprocess, sys
def run(*args):
    subprocess.run([str(arg) for arg in args], check=True)
run(sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt',
    'gdown>=5.2,<6', 'onnx>=1.16', 'onnxruntime>=1.18')
run('apt-get', 'update', '-qq')
run('apt-get', 'install', '-y', '-qq', 'libarchive-tools')
# Print the child traceback explicitly so setup failures are visible in Colab.
asset_result = subprocess.run([sys.executable, 'scripts/download_assets.py'],
                              stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
print(asset_result.stdout, flush=True)
asset_result.check_returncode()
environment = subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True)
(DRIVE_ROOT / 'environment.txt').write_text(environment)
DATA = Path('/content/driver_data')


## Prepare and train YawDD first
This provides an initial trained artifact before the much larger UTA download.
The script excludes unlabeled Dash videos. Talking-and-yawning videos are
positive bags because they contain a yawn; their individual frames are not labeled positive.
Check `metadata.json` in the cached preparation for exclusions and crop failures.


In [ ]:
run(sys.executable, 'scripts/colab_prepare.py', '--task', 'yawn',
    '--drive-root', DRIVE_ROOT, '--work', DATA, '--yawdd', YAWDD_PATH)
def train(task):
    output = DRIVE_ROOT / 'training' / task
    if (output / 'best.pt').is_file() and (output / 'test_metrics.json').is_file():
        print(f'Reusing completed {task} model:', output / 'best.pt')
        return output
    args = [sys.executable, 'scripts/train_video.py', '--task', task,
            '--data', DATA / task, '--output', output, '--epochs', EPOCHS,
            '--batch-size', BATCH_SIZE, '--workers', '2', '--device', 'cuda']
    if (output / 'last.pt').is_file():
        args += ['--resume']
    elif output.exists():
        raise RuntimeError(f"Incomplete run before its first checkpoint: {output}. Rename that directory and rerun.")
    run(*args)
    return output
yawn_run = train('yawn')


## Train the smaller MRL eye-state branch and bundle both models
MRL supplies open/closed eye supervision. The live interface fuses sustained
eye closure with the separately trained YawDD yawn evidence.


In [ ]:
"""Prepare MRL Eyes locally, train visibly, resume safely, and bundle both models."""
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
import time
import urllib.request
import zipfile
import torch
from IPython.display import display


def run_training(command):
    """Use unbuffered output so every batch-progress line appears in Colab."""
    print('Starting eye training on:', torch.cuda.get_device_name(0), flush=True)
    environment = dict(os.environ, PYTHONUNBUFFERED='1')
    subprocess.run([str(value) for value in command], check=True, env=environment)

MRL_URL = 'https://mrl.cs.vsb.cz/data/eyedataset/mrlEyes_2018_01.zip'
drive_source = DRIVE_ROOT / 'mrlEyes_2018_01.zip'
local_source = Path('/content/driver_sources/mrlEyes_2018_01.zip')
source = drive_source if drive_source.is_file() else local_source
prepared = DATA / 'eye_state'
eye_run = DRIVE_ROOT / 'training' / 'eye_state_v2'
prepared_cache = DRIVE_ROOT / 'prepared_v2' / 'eye_state.zip'

if not (prepared / 'metadata.json').is_file():
    if prepared_cache.is_file():
        print('Restoring prepared MRL dataset cache from Drive...', flush=True)
        prepared.parent.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(prepared_cache) as archive:
            archive.extractall(prepared.parent)
        if not (prepared / 'metadata.json').is_file():
            raise RuntimeError('Prepared dataset cache is incomplete. Delete it and rerun.')
        print('Prepared MRL dataset restored.', flush=True)
    else:
        local_source.parent.mkdir(parents=True, exist_ok=True)
        if drive_source.is_file():
            print('Using MRL Eyes archive from Drive:', drive_source, flush=True)
        elif not local_source.is_file():
            print('Downloading official MRL Eyes archive (~342 MB)...', flush=True)
            temporary = local_source.with_suffix('.zip.part')
            last_reported = [-1]
            def report_download(blocks, block_size, total):
                if total > 0:
                    percent = min(100, int(blocks * block_size * 100 / total))
                    if percent // 10 != last_reported[0]:
                        print(f'MRL download: {percent}%', flush=True)
                        last_reported[0] = percent // 10
            urllib.request.urlretrieve(MRL_URL, temporary, reporthook=report_download)
            with zipfile.ZipFile(temporary) as archive:
                bad = archive.testzip()
            if bad:
                raise RuntimeError(f'MRL ZIP integrity check failed at {bad}. Rerun this cell.')
            temporary.replace(local_source)
            source = local_source
        else:
            source = local_source
            print('Using verified local MRL archive:', source, flush=True)
        with zipfile.ZipFile(source) as archive:
            bad = archive.testzip()
        if bad:
            raise RuntimeError(f'MRL ZIP integrity check failed at {bad}. Upload/download it again.')
        run(sys.executable, 'scripts/prepare_dataset.py', '--source', source, '--output', prepared)
        prepared_cache.parent.mkdir(parents=True, exist_ok=True)
        temporary_cache = prepared_cache.with_suffix('.zip.part')
        print('Caching prepared MRL dataset to Drive for future sessions...', flush=True)
        with zipfile.ZipFile(temporary_cache, 'w', zipfile.ZIP_STORED) as archive:
            for path in prepared.rglob('*'):
                if path.is_file():
                    archive.write(path, path.relative_to(prepared.parent))
        temporary_cache.replace(prepared_cache)
        print('Prepared dataset cache saved:', prepared_cache, flush=True)
else:
    print('Using prepared MRL eye dataset from this runtime.')

eye_complete = (eye_run / 'best.pt').is_file() and (eye_run / 'test_metrics.json').is_file()
if not eye_complete:
    command = [sys.executable, 'scripts/train.py', '--data', prepared, '--output', eye_run,
               '--epochs', '15', '--freeze-epochs', '1', '--batch-size', '128',
               '--workers', '4', '--patience', '4', '--device', 'cuda']
    if (eye_run / 'last.pt').is_file():
        print('Resuming interrupted eye-state training from last.pt.', flush=True)
        run_training(command + ['--resume'])
    else:
        if eye_run.exists():
            suffix = 1
            backup = eye_run.with_name(f'{eye_run.name}_incomplete_{suffix}')
            while backup.exists():
                suffix += 1
                backup = eye_run.with_name(f'{eye_run.name}_incomplete_{suffix}')
            eye_run.rename(backup)
            print('Preserved incomplete eye run at:', backup)
        print('Starting the new clean eye-state training run.', flush=True)
        run_training(command)
else:
    print('Using completed eye-state checkpoint:', eye_run / 'best.pt')

outputs = {'eye_state': eye_run, 'yawn': DRIVE_ROOT / 'training' / 'yawn'}
for task, output in outputs.items():
    if not (output / 'best.pt').is_file() or not (output / 'test_metrics.json').is_file():
        raise RuntimeError(f'{task} training is incomplete: {output}')
    print(task.upper())
    display(json.loads((output / 'test_metrics.json').read_text()))

bundle = DRIVE_ROOT / 'trained_models.zip'
temporary_bundle = bundle.with_suffix('.zip.part')
with zipfile.ZipFile(temporary_bundle, 'w', zipfile.ZIP_DEFLATED) as archive:
    for task in ('eye_state', 'yawn'):
        folder = outputs[task]
        for name in ('best.pt', 'config.json', 'test_metrics.json', 'history.json'):
            if (folder / name).is_file():
                archive.write(folder / name, f'{task}/{name}')
        for name in ('dataset_report.json', 'status.json', 'encoder.onnx', 'encoder.json'):
            if (folder / name).is_file():
                archive.write(folder / name, f'{task}/{name}')
temporary_bundle.replace(bundle)
print('Saved smaller-pipeline models:', bundle)
from google.colab import files
files.download(str(bundle))


## Open the Windows live interface
Extract `trained_models.zip` into the project's `models/trained/` directory.
It should contain `eye_state/best.pt` and `yawn/best.pt`.
Run `python app/server.py`, open **http://127.0.0.1:8765**, and click **Start camera**.
Allow about 16 seconds to fill the first model window. If the interface was already
running, click **Reload models**.

References: [MRL Eyes](https://mrl.cs.vsb.cz/eyedataset.html),
[YawDD](https://doi.org/10.21227/e1qm-hb90),
[YuNet](https://github.com/opencv/opencv_zoo/tree/main/models/face_detection_yunet).
